## Web Scraping Obstruction Evaluation/Airport Airspace Analysis (OE/AAA website)

https://oeaaa.faa.gov/oeaaa/oe3a/main/#/search/records

---

#### Scenario 1: Given the FCC-ASR number, give me the latest ASN (FAA registry number) that will show up from the FAA's website.

In [1]:
import requests
import pandas as pd
import json
import datetime
import re

api_endpoint = "https://oeaaa.faa.gov/oeaaa/tools-api/namedOperation.do"

headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
}

api_payload_template = {
    "areaType": "id",
    "timeSpan": "120",
    "formLat": "",
    "formLon": "",
    "radiusNM": 25,
    "structureType": "ANY",
    "allStatusSelected": True,
    "criteria": {},
    "fcc": "",
    "opName": "GET_CASE_BY_FCC",
    "placement": "OFF_AIRPORT",
    "status": {},
    "structureTypes": ["ANY"]
}

asn_key_in_response = "ASN"

data_key_in_response = "SUBMITTED_DATE"

def get_asn_via_api(asr_numbers):

    results_data = []

    for asr in asr_numbers:
        asn = 'N/A'
        status = 'Processing'

        payload = api_payload_template.copy()

        if "fcc" in payload:
            payload["fcc"] = str(asr)
        else:
            status = 'Configuration Error: "fcc" key not in payload template'

        if status != 'Processing':
            results_data.append({'FCC_ASR': asr, 'ASN': asn, 'Status': status})
            continue

        try:
            print(f"Querying API for ASR: {asr}...")
            response = requests.post(
                api_endpoint,
                headers = headers,
                data = json.dumps(payload),
                timeout = 15
            )

            if response.status_code == 200:
                data = response.json()

                if data and isinstance(data, list) and len(data) > 0:

                    def parse_date(record):
                        date_str = record.get(data_key_in_response)
                        if date_str and date_str != "N/A":
                            try:
                                return datetime.datetime.strptime(date_str, "%m/%d/%Y")
                            except ValueError:
                                return datetime.datetime.min
                        return datetime.datetime.min

                    latest_record = max(data, key = parse_date)


                    if asn_key_in_response in latest_record:
                        asn = latest_record[asn_key_in_response]
                        status = f'Success (Latest of {len(data)} records)'
                    else:
                        status = f'Success, but ASN key "{asn_key_in_response}" not found in latest record.'

                else:
                    status = 'No Records Found (API returned [])'
            else:
                status = f"HTTP Error: {response.status_code}"

        except requests.exceptions.Timeout:
            status = 'Request Timeout'
        except requests.exceptions.RequestException as e:
            status = f'Connection Error: {e}'
        except json.JSONDecodeError:
            status = 'API returned non-JSON response (Likely HTML/Error Page)'
        except Exception as e:
            status = f'Unexpected Error: {type(e).__name__}'

        print(f"  -> Result Status: {status}")
        results_data.append({'FCC_ASR': asr, 'ASN': asn, 'Status': status})

    return pd.DataFrame(results_data)


# sample
input_asr_numbers = [1305469, 1002875, 1212593, 1212906]

final_df = get_asn_via_api(input_asr_numbers)

print("\n" + "="*50)
print("    FINAL FAA ASR to ASN Mapping (API Method)")
print("="*50)
print(final_df.to_markdown(index=False))
print("="*50)

Querying API for ASR: 1305469...
  -> Result Status: No Records Found (API returned [])
Querying API for ASR: 1002875...
  -> Result Status: Success (Latest of 1 records)
Querying API for ASR: 1212593...
  -> Result Status: Success (Latest of 2 records)
Querying API for ASR: 1212906...
  -> Result Status: Success (Latest of 2 records)

    FINAL FAA ASR to ASN Mapping (API Method)
|   FCC_ASR | ASN               | Status                             |
|----------:|:------------------|:-----------------------------------|
|   1305469 | N/A               | No Records Found (API returned []) |
|   1002875 | 2000-AEA-4199-OE  | Success (Latest of 1 records)      |
|   1212593 | 2021-AGL-15579-OE | Success (Latest of 2 records)      |
|   1212906 | 2025-AEA-4815-OE  | Success (Latest of 2 records)      |


In [2]:
final_df

,FCC_ASR,ASN,Status
0,1305469,N/A,No Records Found (API returned [])
1,1002875,2000-AEA-4199-OE,Success (Latest of 1 records)
2,1212593,2021-AGL-15579-OE,Success (Latest of 2 records)
3,1212906,2025-AEA-4815-OE,Success (Latest of 2 records)


---

#### Scenario 2: Given the FAA registry number, give me the FCC-ASR that will show up from the hyperlink of the ASN as seen in FAA's UI.

In [2]:
api_endpoint = "https://oeaaa.faa.gov/oeaaa/oe3a/external/portal-api/caseFiling/dynamicCaseDataByAsn.do" 

headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
}

api_payload_template = {
    "areaType": "id",
    "timeSpan": "120",
    "formLat": "",
    "formLon": "",
    "radiusNM": 25,
    "structureType": "ANY",
    "allStatusSelected": True,
    "criteria": {}, 
    "asnRegion": "",    
    "asnYear": 0,       
    "asnSequence": "", 
    "asnCaseType": "",  
    "placement": "OFF_AIRPORT",
    "status": {},
    "structureTypes": ["ANY"]
}

fcc_asr_key_in_response = "fccAsr" 

date_key_in_response = "submittedOn" 


def parse_asn(asn_string):

    match = re.match(r"(\d{4})-([A-Z]{3})-([\d\w]+)-([A-Z]{2})", asn_string)
    
    if match:
        year, region, sequence, casetype = match.groups()
        return {
            "asnYear": int(year),
            "asnRegion": region,
            "asnSequence": sequence,
            "asnCaseType": casetype
        }
    return None

def get_asr_via_api(asn_numbers):

    results_data = []

    for asn in asn_numbers:
        asr = 'N/A'
        status = 'Processing'
        
        asn_parts = parse_asn(str(asn))
        if not asn_parts:
            status = 'Input Error: Invalid ASN format'
            results_data.append({'ASN': asn, 'FCC_ASR': asr, 'Status': status})
            continue

        payload = api_payload_template.copy()
        payload.update(asn_parts)

        try:
            print(f"Querying API for ASN: {asn}...")
            response = requests.post(
                api_endpoint, 
                headers=headers, 
                data=json.dumps(payload), 
                timeout=15
            )
            
            if response.status_code == 200:
                data = response.json()
                
                if isinstance(data, dict) and data:
                    record = data
                    
                    if fcc_asr_key_in_response in record: 
                        asr = record[fcc_asr_key_in_response]
                        status = 'Success (Found single record)'
                    else:
                        status = f'Success, but FCC-ASR key "{fcc_asr_key_in_response}" not found in record.'
                
                elif isinstance(data, list) and len(data) > 0:
                    first_record = data[0]
                    if fcc_asr_key_in_response in first_record: 
                        asr = first_record[fcc_asr_key_in_response]
                        status = f'Success (List received, taking first of {len(data)})'
                    else:
                        status = f'Success, but FCC-ASR key "{fcc_asr_key_in_response}" not found in list record.'
                
                else:
                    status = 'No Records Found (API returned empty/unrecognized structure)'
            else:
                status = f"HTTP Error: {response.status_code}"

        except requests.exceptions.Timeout:
            status = 'Request Timeout'
        except requests.exceptions.RequestException as e:
            status = f"Connection Error: {e}"
        except json.JSONDecodeError:
            status = 'API returned non-JSON response (Likely HTML/Error Page)'
        except Exception as e:
            status = f"Unexpected Error: {type(e).__name__}"

        print(f"  -> Result Status: {status}")
        results_data.append({'ASN': asn, 'FCC_ASR': asr, 'Status': status})

    return pd.DataFrame(results_data)


input_asn_numbers = ["2018-AEA-351-OE", "2000-AEA-4199-OE", "2021-AGL-15579-OE", "2025-AEA-4815-OE"]

final_df = get_asr_via_api(input_asn_numbers)

print("\n" + "="*50)
print("       FAA ASN to FCC-ASR Mapping (API Method)")
print("="*50)
print(final_df.to_markdown(index=False))
print("="*50)

Querying API for ASN: 2018-AEA-351-OE...
  -> Result Status: Success (Found single record)
Querying API for ASN: 2000-AEA-4199-OE...
  -> Result Status: Success (Found single record)
Querying API for ASN: 2021-AGL-15579-OE...
  -> Result Status: Success (Found single record)
Querying API for ASN: 2025-AEA-4815-OE...
  -> Result Status: Success (Found single record)

       FAA ASN to FCC-ASR Mapping (API Method)
| ASN               |   FCC_ASR | Status                        |
|:------------------|----------:|:------------------------------|
| 2018-AEA-351-OE   |   1008239 | Success (Found single record) |
| 2000-AEA-4199-OE  |   1002875 | Success (Found single record) |
| 2021-AGL-15579-OE |   1212593 | Success (Found single record) |
| 2025-AEA-4815-OE  |   1212906 | Success (Found single record) |
